# EngSVG - LoRA SFT on engineering drawings, scored with FEM

**VS Code + Colab extension:** Select Kernel > Colab > Auto Connect, then Run All.
The kernel is a Colab GPU VM; outputs are saved into this local file.

**Colab web:** Runtime > Change runtime type > GPU, then Run all.

Results are printed inline (so they persist in this file) and written to
`/content/engsvg-run/` on the VM.

In [2]:
import subprocess, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU - select a GPU runtime')

NVIDIA A100-SXM4-40GB, 40960 MiB


In [3]:
!rm -rf /content/SVG
!git clone -q --depth 1 -b cvpr2027-research-package --filter=blob:none --sparse \
    https://github.com/ayushdebnath012/SVG.git /content/SVG
!cd /content/SVG && git sparse-checkout set cvpr2027/scripts cvpr2027/src \
    cvpr2027/data/eng-svg-bench-text cvpr2027/data/eng-svg-bench-edit
!pip -q install svgpathtools==1.7.2 peft transformers accelerate scikit-fem
# Colab preinstalls torchao 0.10; peft's LoRA dispatch raises on anything below 0.16.
!pip uninstall -y -q torchao
!cd /content/SVG && git log --oneline -1

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 15 (delta 0), reused 12 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 4.69 MiB | 18.46 MiB/s, done.
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 140 (delta 50), reused 86 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 278.50 KiB | 3.52 MiB/s, done.
Resolving deltas: 100% (50/50), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.5/178.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 6.3 MB/s eta 0:00:00
10f9cf8 (grafted, HEAD -> cvpr2027-research-package, origin/cvpr2027-research-package) Add a runnable notebook for the EngSVG training run


In [4]:
# Regenerate the 800 training pairs deterministically (seed 20260923), then verify them.
!cd /content/SVG/cvpr2027 && PYTHONPATH=src:scripts python scripts/eng_svg_trainset.py build --n 400
!cd /content/SVG/cvpr2027 && PYTHONPATH=src:scripts python scripts/eng_svg_trainset.py verify

48 frozen benchmark geometries excluded
  text2svg: 50/400
  text2svg: 100/400
  text2svg: 150/400
  text2svg: 200/400
  text2svg: 250/400
  text2svg: 300/400
  text2svg: 350/400
  text2svg: 400/400
  svg2svg: 50/400
  svg2svg: 100/400
  svg2svg: 150/400
  svg2svg: 200/400
  svg2svg: 250/400
  svg2svg: 300/400
  svg2svg: 350/400
  svg2svg: 400/400
{
  "version": "eng-svg-trainset-v1",
  "seed": 20260923,
  "per_arm": 400,
  "pairs": 800,
  "splits": {
    "train": 639,
    "validation": 81,
    "test": 80
  },
  "rejected_before_acceptance": 64,
  "frozen_geometries_excluded": 48,
  "split_rule": "by geometry key, so near-duplicate frames cannot straddle splits",
  "acceptance": "every target was scored by cad_astra_benchmark.score and written only if clean, and only if a FEM re-solve of its reconstructed drawn geometry matched the reference peak stress to 2e-3",
  "style_randomisation": [
    "member element type (line/polyline/path)",
    "ink and accent colour",
    "stroke width",


In [5]:
# Base vs trained on 24 held-out text2svg tasks, scored by the real benchmark scorer.
!cd /content/SVG/cvpr2027 && PYTHONPATH=src:scripts ENGSVG_ROOT=/content/SVG/cvpr2027 \
    python scripts/colab_train_engsvg.py --arms text2svg --epochs 2 --eval-count 24 \
    --out /content/engsvg-run

commit 10f9cf8 | torch 2.11.0+cu128 | NVIDIA A100-SXM4-40GB
311 train / 44 val / 45 test  arms=('text2svg',)
config.json: 100% 660/660 [00:00<00:00, 2.60MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 13.0MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 44.4MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 108MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 156MB/s]
311 encoded train examples, max len 4096

model.safetensors: downloading bytes:   9% 287M/3.09G [00:01<00:11, 243MB/s, 24.3MB/s  ]  
model.safetensors: downloading bytes:  15% 455M/3.09G [00:02<00:12, 206MB/s, 39.0MB/s  ]
model.safetensors: downloading bytes:  17% 515M/3.09G [00:02<00:12, 207MB/s, 42.6MB/s  ]  ]
model.safetensors: downloading bytes:  75% 2.32G/3.09G [00:16<02:04, 6.17MB/s, 78.3MB/s  ] 
model.safetensors: downloading bytes:  78% 2.40G/3.09G [01:05<05:12, 2.18MB/s, 3.74MB/s  ]
model.safetensors: downloading bytes:  82% 2.54G/3.09G [01:06<00:03, 179MB/s, 10.6MB/s  ]  ]
model.safetensors: downloa

: 

In [ ]:
# Print the full summary inline so it is captured in this notebook file.
import json, pathlib
s = json.loads(pathlib.Path('/content/engsvg-run/summary.json').read_text())
for tag in ('before','after'):
    r = s[tag]; n = max(r['n'], 1)
    print(f"{r['tag']:8s} parsed {r['parsed']:2d}/{r['n']}  geometry {r['geometry_ok']:2d}/{n}  "
          f"dimensions {r['dimensions_ok']:2d}/{n}  analysis {r['analysis_ok']:2d}/{n}  "
          f"drawnFEM {r['drawn_fem_ok']:2d}/{n}  STRICT {r['strict_pass']:2d}/{n}")
print()
print(json.dumps({k: v for k, v in s.items() if k not in ('before','after')}, indent=2))